# Quick Test Validation: Short ncRNA Overlap with Mouse Annotation

Check whether predicted short ncRNA intervals (projected from human to mouse)
overlap known GENCODE mouse exons on the quick test chromosomes (chrX, chr7, chr19).

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pyrion_path = Path("/Users/Bogdan.Kirilenko/PycharmProjects/pyrion")
if pyrion_path not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(pyrion_path))

from pyrion.io.bed import read_narrow_bed_file, read_bed12_file
from pyrion.ops.interval_ops import intersect_intervals

In [ ]:
PIPELINE_DIR = Path("../quick_test")
ANNOTATION_DIR = Path("../input_data/mm39_annotation_validation")

pred_bed = PIPELINE_DIR / "intermediate_bed_files" / "short_rna_annotation_intermediate.bed"
anno_bed = ANNOTATION_DIR / "mm39_quick_test_transcripts.bed"
trna_bed = ANNOTATION_DIR / "mm39-tRNAs.bed"
joblist_path = PIPELINE_DIR / "joblists" / "short_ncRNA_joblist.txt"

predictions = read_narrow_bed_file(str(pred_bed))
annotation = read_bed12_file(str(anno_bed))
trna_annotation = read_bed12_file(str(trna_bed))
joblist = pd.read_csv(joblist_path, sep="\t")

# scores from BED column 5 (not exposed by read_narrow_bed_file)
_bed_df = pd.read_csv(pred_bed, sep="\t", header=None, usecols=[3, 4], names=["pred_id", "score"])
score_map = dict(zip(_bed_df["pred_id"], _bed_df["score"]))

print(f"TOGA jobs:    {len(joblist)}")
print(f"Predictions: {len(predictions)} intervals")
print(f"GENCODE:     {len(annotation):,} transcripts")
print(f"tRNAs:       {len(trna_annotation):,} entries")

In [ ]:
rows = []
for pred in predictions:
    pred_arr = np.array([[pred.start, pred.end]])
    pred_len = pred.end - pred.start

    # GENCODE overlap
    hits = annotation.get_transcripts_in_interval(pred)
    best_bp = 0
    best_tid = None
    n_exon_hits = 0
    for t in hits:
        isect = intersect_intervals(pred_arr, t.blocks)
        if len(isect) > 0:
            bp = int(np.sum(isect[:, 1] - isect[:, 0]))
            n_exon_hits += 1
            if bp > best_bp:
                best_bp = bp
                best_tid = t.id

    # tRNA overlap
    trna_hits = trna_annotation.get_transcripts_in_interval(pred)
    best_trna_bp = 0
    best_trna_id = None
    for t in trna_hits:
        isect = intersect_intervals(pred_arr, t.blocks)
        if len(isect) > 0:
            bp = int(np.sum(isect[:, 1] - isect[:, 0]))
            if bp > best_trna_bp:
                best_trna_bp = bp
                best_trna_id = t.id

    combined_bp = max(best_bp, best_trna_bp)
    pid = pred.id or f"{pred.chrom}:{pred.start}-{pred.end}"
    rows.append({
        "pred_id": pid,
        "chrom": pred.chrom,
        "start": pred.start,
        "end": pred.end,
        "length": pred_len,
        "score": score_map.get(pid, np.nan),
        "gencode_overlap_bp": best_bp,
        "gencode_overlap_pct": best_bp / pred_len * 100 if pred_len > 0 else 0,
        "trna_overlap_bp": best_trna_bp,
        "trna_overlap_pct": best_trna_bp / pred_len * 100 if pred_len > 0 else 0,
        "best_overlap_bp": combined_bp,
        "best_overlap_pct": combined_bp / pred_len * 100 if pred_len > 0 else 0,
        "n_transcript_span_hits": len(hits),
        "n_exon_hits": n_exon_hits,
        "n_trna_hits": len(trna_hits),
        "best_transcript": best_tid,
        "best_trna": best_trna_id,
    })

df = pd.DataFrame(rows)
df["has_exon_overlap"] = df["gencode_overlap_bp"] > 0
df["has_trna_overlap"] = df["trna_overlap_bp"] > 0
df["has_any_overlap"] = df["best_overlap_bp"] > 0
df.head()

In [ ]:
joblist_ids = set(joblist["transcript_id"] + "." + joblist["chain_id"].astype(str))
pred_ids = set(df["pred_id"])
n_jobs = len(joblist_ids)
n_with_pred = len(joblist_ids & pred_ids)

print(f"=== TOGA Joblist Coverage ===")
print(f"TOGA jobs:         {n_jobs}")
print(f"With prediction:   {n_with_pred} ({n_with_pred/n_jobs*100:.1f}%)")
print(f"No output:         {n_jobs - n_with_pred}")

n_total = len(df)
n_gencode = df["has_exon_overlap"].sum()
n_trna_only = (df["has_trna_overlap"] & ~df["has_exon_overlap"]).sum()
n_any = df["has_any_overlap"].sum()
n_miss = n_total - n_any
n_intron_only = ((df["n_transcript_span_hits"] > 0) & ~df["has_any_overlap"]).sum()
n_no_transcript = n_miss - n_intron_only

print(f"\n=== Overlap Summary ===")
print(f"Total predictions:          {n_total}")
print(f"GENCODE exon overlap:       {n_gencode} ({n_gencode/n_total*100:.1f}%)")
print(f"tRNA-only overlap:          {n_trna_only} ({n_trna_only/n_total*100:.1f}%)")
print(f"Any overlap (GENCODE+tRNA): {n_any} ({n_any/n_total*100:.1f}%)")
print(f"No overlap at all:          {n_miss} ({n_miss/n_total*100:.1f}%)")
print(f"  - in transcript intron:   {n_intron_only}")
print(f"  - no transcript at all:   {n_no_transcript}")

hits = df[df["has_any_overlap"]]
if len(hits) > 0:
    print(f"\nAmong {n_any} overlapping:")
    print(f"  Overlap %:  median={hits['best_overlap_pct'].median():.1f}%  "
          f"mean={hits['best_overlap_pct'].mean():.1f}%")
    full = (hits["best_overlap_pct"] >= 99.0).sum()
    print(f"  Full (≥99%): {full} ({full/n_total*100:.1f}% of all)")

print(f"\nPer chromosome:")
for chrom in sorted(df["chrom"].unique()):
    sub = df[df["chrom"] == chrom]
    h = sub["has_any_overlap"].sum()
    print(f"  {chrom:6s}: {len(sub):4d} total, {h:4d} overlap ({h/len(sub)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Overlap breakdown
ax = axes[0]
categories = ["GENCODE\nexon", "tRNA\nonly", "Intron\nonly", "No\nannotation"]
values = [n_gencode, n_trna_only, n_intron_only, n_no_transcript]
colors = ["#4CAF50", "#66BB6A", "#FFA726", "#E57373"]
bars = ax.bar(categories, values, color=colors, edgecolor="white", width=0.55)
for bar, v in zip(bars, values):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(v), ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Count")
ax.set_title("Prediction overlap categories")

# 2. Overlap % histogram
ax = axes[1]
if len(hits) > 0:
    ax.hist(hits["best_overlap_pct"], bins=20, range=(0, 100),
            color="#42A5F5", edgecolor="white")
    med = hits["best_overlap_pct"].median()
    ax.axvline(med, color="red", linestyle="--", label=f"median={med:.0f}%")
    ax.legend()
ax.set_xlabel("Best overlap % (GENCODE or tRNA)")
ax.set_ylabel("Count")
ax.set_title("Overlap % distribution (hits only)")

# 3. Per chromosome stacked bars
ax = axes[2]
chrom_stats = df.groupby("chrom")["has_any_overlap"].agg(["sum", "count"])
chrom_stats.columns = ["overlap", "total"]
chrom_stats["no_overlap"] = chrom_stats["total"] - chrom_stats["overlap"]
chrom_stats[["overlap", "no_overlap"]].plot.bar(
    stacked=True, ax=ax, color=["#4CAF50", "#E57373"], edgecolor="white")
ax.set_ylabel("Count")
ax.set_title("Per chromosome")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(["Overlap", "No overlap"], loc="upper right")

plt.tight_layout()
plt.show()

## Score analysis

Does score=1000 correlate with higher overlap?

In [ ]:
df["score_1000"] = df["score"] == 1000

hit_scores = df[df["has_any_overlap"]]["score"]
miss_scores = df[~df["has_any_overlap"]]["score"]

print(f"=== Score Summary ===")
print(f"Score range: {df['score'].min():.0f} – {df['score'].max():.0f}")
print(f"Score=1000:  {df['score_1000'].sum()} / {n_total} ({df['score_1000'].mean()*100:.1f}%)")
print()
print(f"{'':30s} {'Any overlap':>14s} {'No overlap':>14s}")
print(f"{'Mean score':30s} {hit_scores.mean():14.1f} {miss_scores.mean():14.1f}")
print(f"{'Median score':30s} {hit_scores.median():14.0f} {miss_scores.median():14.0f}")
print(f"{'Fraction with score=1000':30s} "
      f"{(hit_scores == 1000).mean()*100:13.1f}% "
      f"{(miss_scores == 1000).mean()*100:13.1f}%")
print()

bins = [0, 700, 800, 900, 999, 1001]
labels = ["<700", "700-799", "800-899", "900-999", "1000"]
df["score_bin"] = pd.cut(df["score"], bins=bins, labels=labels, right=False)

print("Overlap rate by score bucket (GENCODE + tRNA):")
for label in labels:
    sub = df[df["score_bin"] == label]
    if len(sub) == 0:
        continue
    rate = sub["has_any_overlap"].mean() * 100
    print(f"  {label:>10s}: {len(sub):4d} predictions, {rate:5.1f}% overlap")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Score distribution: overlap vs no overlap
ax = axes[0]
ax.hist(hit_scores, bins=30, alpha=0.7, label=f"Any overlap (n={len(hit_scores)})",
        color="#4CAF50", edgecolor="white")
ax.hist(miss_scores, bins=30, alpha=0.7, label=f"No overlap (n={len(miss_scores)})",
        color="#E57373", edgecolor="white")
ax.set_xlabel("Score")
ax.set_ylabel("Count")
ax.set_title("Score distribution")
ax.legend(fontsize=8)

# 2. Overlap rate by score bucket
ax = axes[1]
bucket_stats = df.groupby("score_bin", observed=False)["has_any_overlap"].agg(["mean", "count"])
bar_x = range(len(bucket_stats))
bars = ax.bar(bar_x, bucket_stats["mean"] * 100, color="#42A5F5", edgecolor="white", width=0.6)
ax.set_xticks(bar_x)
ax.set_xticklabels(bucket_stats.index, fontsize=9)
for i, (rate, count) in enumerate(zip(bucket_stats["mean"], bucket_stats["count"])):
    if count > 0:
        ax.text(i, rate * 100 + 1.5, f"n={count}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Score bucket")
ax.set_ylabel("Overlap rate (%)")
ax.set_title("Overlap rate by score")
ax.set_ylim(0, 105)

# 3. Score vs overlap %
ax = axes[2]
ax.scatter(df["score"], df["best_overlap_pct"], alpha=0.5, s=20,
           c=df["has_any_overlap"].map({True: "#4CAF50", False: "#E57373"}),
           edgecolors="none")
ax.set_xlabel("Score")
ax.set_ylabel("Best overlap (%)")
ax.set_title("Score vs overlap %")

plt.tight_layout()
plt.show()

In [ ]:
misses = df[~df["has_any_overlap"]].copy()
if len(misses) > 0:
    print(f"=== {len(misses)} predictions without any overlap (GENCODE or tRNA) ===\n")
    print(misses[["pred_id", "chrom", "start", "end", "length",
                   "score", "n_transcript_span_hits"]].to_string(index=False))
else:
    print("All predictions overlap an annotated exon or tRNA.")

In [ ]:
suspicious = df[(df["score"] == 1000) & ~df["has_any_overlap"]]
print(f"=== {len(suspicious)} predictions with score=1000 but no overlap (GENCODE or tRNA) ===\n")
print(suspicious[["pred_id", "chrom", "start", "end", "length",
                   "n_transcript_span_hits"]].to_string(index=False))

## EST support for non-overlapping predictions

Check if predictions that missed GENCODE + tRNA have any EST alignment evidence.
Source: UCSC `all_est` table (binned PSL format).

In [ ]:
est_path = ANNOTATION_DIR / "all_est.txt"

# binned PSL columns: 14=tName, 16=tStart, 17=tEnd, 19=blockSizes, 21=tStarts
QUICK_TEST_CHROMS = {"chr7", "chr19", "chrX"}
chunks = []
for chunk in pd.read_csv(est_path, sep="\t", header=None,
                          usecols=[14, 16, 17, 19, 21], chunksize=500_000):
    chunk.columns = ["chrom", "tStart", "tEnd", "blockSizes", "tStarts"]
    chunks.append(chunk[chunk["chrom"].isin(QUICK_TEST_CHROMS)])

est_df = pd.concat(chunks, ignore_index=True)
print(f"Loaded {len(est_df):,} EST alignments on {', '.join(sorted(QUICK_TEST_CHROMS))}")

In [ ]:
def parse_est_blocks(row):
    sizes = [int(x) for x in row["blockSizes"].rstrip(",").split(",") if x]
    starts = [int(x) for x in row["tStarts"].rstrip(",").split(",") if x]
    return np.array([[s, s + sz] for s, sz in zip(starts, sizes)])

misses = df[~df["has_any_overlap"]].copy()
est_results = []

for _, miss in misses.iterrows():
    chrom_ests = est_df[est_df["chrom"] == miss["chrom"]]
    span_hits = chrom_ests[
        (chrom_ests["tStart"] < miss["end"]) & (chrom_ests["tEnd"] > miss["start"])
    ]
    n_block_hits = 0
    if len(span_hits) > 0:
        pred_arr = np.array([[miss["start"], miss["end"]]])
        for _, est in span_hits.iterrows():
            blocks = parse_est_blocks(est)
            if len(intersect_intervals(pred_arr, blocks)) > 0:
                n_block_hits += 1

    in_intron = miss["n_transcript_span_hits"] > 0
    est_results.append({
        "pred_id": miss["pred_id"],
        "score": miss["score"],
        "location": "intron" if in_intron else "desert",
        "n_est_span": len(span_hits),
        "n_est_block": n_block_hits,
        "est_support": "yes" if n_block_hits > 0 else "no",
    })

est_df_results = pd.DataFrame(est_results)

print(f"=== EST support for {len(misses)} non-overlapping predictions ===\n")

ct = est_df_results.groupby(["location", "est_support"]).size().unstack(fill_value=0)
for loc in ["intron", "desert"]:
    if loc not in ct.index:
        continue
    row = ct.loc[loc]
    total = row.sum()
    y = row.get("yes", 0)
    n = row.get("no", 0)
    print(f"  {loc:7s}: {total:3d} total  |  EST support: {y:3d}  |  no support: {n:3d}")

print()
print(est_df_results.to_string(index=False))

In [ ]:
desert = est_df_results[est_df_results["location"] == "desert"].merge(
    misses[["pred_id", "chrom", "start", "end"]], on="pred_id")

print(f"=== {len(desert)} predictions in gene desert ===\n")
for _, r in desert.iterrows():
    print(f"{r['chrom']}:{r['start']}-{r['end']}  {r['pred_id']}  score={r['score']}  est={r['n_est_block']}")